# Scaled Dot-Product Attention from Scratch

> Part of the [ML Notebooks](../README.md) series — by **Nandobez**.


## Intuition

Attention lets every token look at every other token and aggregate information weighted by *relevance*. The query asks a question, the keys advertise content, the values are what gets summed up. The scaling by $\sqrt{d_k}$ keeps the dot-products small so the softmax does not saturate.


## Mathematical Formulation

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

with $Q \in \mathbb{R}^{n \times d_k}$, $K \in \mathbb{R}^{m \times d_k}$, $V \in \mathbb{R}^{m \times d_v}$.

- Each row of $QK^\top$ is the unnormalised similarity between one query and all keys.
- Softmax over the last dimension turns those scores into a probability distribution.
- Multiplying by $V$ produces a weighted average of value vectors.


## Implementation


In [ ]:
import math
import numpy as np
import torch
import torch.nn.functional as F

torch.manual_seed(0)


In [ ]:
def attention_numpy(Q, K, V, mask=None):
    """Reference implementation in pure NumPy."""
    d_k = Q.shape[-1]
    scores = Q @ K.swapaxes(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    weights = np.exp(scores - scores.max(axis=-1, keepdims=True))
    weights = weights / weights.sum(axis=-1, keepdims=True)
    return weights @ V, weights


In [ ]:
def attention_torch(Q, K, V, mask=None):
    """Same thing with autograd-friendly tensors."""
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(~mask, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights


## Experiment


In [ ]:
n, m, d_k, d_v = 4, 6, 8, 8
Q = torch.randn(n, d_k)
K = torch.randn(m, d_k)
V = torch.randn(m, d_v)

out, w = attention_torch(Q, K, V)
print('out shape:', out.shape)
print('attn weights row sums:', w.sum(-1))


In [ ]:
# sanity-check against torch's built-in scaled_dot_product_attention
ref = F.scaled_dot_product_attention(Q.unsqueeze(0), K.unsqueeze(0), V.unsqueeze(0)).squeeze(0)
print('max diff:', (out - ref).abs().max().item())


## Discussion

- The scaling factor $\sqrt{d_k}$ matters: without it the softmax saturates at large $d_k$, producing tiny gradients.
- The mask is what differentiates causal (decoder) from bidirectional (encoder) attention.
- For long sequences, the $O(n^2)$ memory cost is the bottleneck — see *FlashAttention* and *Linear Attention* for fixes.


## References

- Series repo: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Author: [Nandobez](https://github.com/Nandobez)
